# External LLM workflow (Ollama optional + mock)

This notebook shows a **real-world pattern**: call an HTTP API (Ollama on `localhost` if present), otherwise fall back to a **mock** response—no GPU and no extra Python dependencies. The callable is then wrapped with **`SafeRunner`** so RAI checks apply to the same payload shape you would use in production.

In [1]:
import json
import urllib.error
import urllib.request
from typing import Any


def chat_ollama_or_mock(prompt: str, model: str = "qwen2", timeout: float = 2.0) -> dict[str, Any]:
    """Try Ollama /api/generate; on any failure return a deterministic mock."""
    body = json.dumps({"model": model, "prompt": prompt, "stream": False}).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = json.loads(resp.read().decode())
        text = (data.get("response") or "").strip()
        return {"output": text, "source": "ollama", "model": model}
    except (OSError, urllib.error.URLError, urllib.error.HTTPError, json.JSONDecodeError, ValueError):
        return {
            "output": f"[mock LLM] {prompt[:200]}",
            "source": "mock",
            "model": model,
        }


demo = chat_ollama_or_mock("Say hello in one word.")
demo

{'output': '[mock LLM] Say hello in one word.',
 'source': 'mock',
 'model': 'qwen2'}

In [2]:
from oris.integrations import SafeRunner
from oris.rai.policy import PolicyEnforcer

policy = PolicyEnforcer()


def llm_step(payload: dict) -> dict:
    prompt = str(payload.get("query", ""))
    return chat_ollama_or_mock(prompt)


safe_llm = SafeRunner(llm_step, policy=policy)
result = safe_llm.run({"query": "Explain RAI in one sentence."}, include_trace=True)
print(result.output)
print("--- trace flags ---", result.to_run_summary()["trace"][0]["flags"])

{'output': '[mock LLM] Explain RAI in one sentence.', 'source': 'mock', 'model': 'qwen2'}
--- trace flags --- {'kind': 'external_pipeline'}


## Same idea with a YAML pipeline + provider stub

Built-in **`openai`** / **`huggingface`** providers are **stubs** (no network): they only require the named env var to be set. Below we set a **non-secret placeholder** so the file loads; output is deterministic and safe for demos.

In [3]:
import os
from pathlib import Path

from oris import Pipeline


def examples_dir() -> Path:
    here = Path.cwd().resolve()
    if (here / "provider_pipeline.yaml").exists():
        return here
    ex = here / "examples"
    if (ex / "provider_pipeline.yaml").exists():
        return ex
    raise FileNotFoundError("Run from repo root or examples/.")


os.environ.setdefault("OPENAI_API_KEY", "notebook-demo-not-a-real-key")
EX2 = examples_dir()
pipe = Pipeline.from_yaml(EX2 / "provider_pipeline.yaml")
pr = pipe.run({"query": "What is a pipeline?"})
pr.output

{'query': 'What is a pipeline?',
 'output': '[openai:gpt-4] What is a pipeline?'}

In [4]:
import json

print(json.dumps(pr.to_run_summary(), indent=2, default=str))

{
  "run_id": "acddf7fa-56c7-431e-ab3f-196503abb6b5",
  "status": "success",
  "output": {
    "query": "What is a pipeline?",
    "output": "[openai:gpt-4] What is a pipeline?"
  },
  "trace": [
    {
      "step_id": "pipeline_pre_0",
      "component_name": "pipeline_pre_0",
      "status": "success",
      "latency_ms": 0.008,
      "flags": {
        "kind": "pipeline_hook",
        "phase": "pre",
        "index": 0
      }
    },
    {
      "step_id": "step1",
      "component_name": "step1",
      "status": "success",
      "latency_ms": 0.006,
      "flags": {
        "kind": "pipeline_step",
        "step_index": 0,
        "total_steps": 1
      }
    },
    {
      "step_id": "pipeline_post_0",
      "component_name": "pipeline_post_0",
      "status": "success",
      "latency_ms": 0.004,
      "flags": {
        "kind": "pipeline_hook",
        "phase": "post",
        "index": 0
      }
    }
  ]
}
